# Causal Walkthrough: Job Training and 1978 Earnings

This notebook is a reader-friendly walkthrough of the repo's core analysis. The actual implementation still lives in `src/` and `scripts/`; this notebook is here to make the workflow easier to inspect interactively.

## Notebook goals

- Load and prepare the LaLonde benchmark data.
- Compare simple propensity specifications.
- Estimate effects with several causal estimators.
- Check placebo outcomes and subgroup heterogeneity.
- Reuse the package code rather than reimplementing it inline.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from causal_eval.diagnostics import covariate_balance_table, effective_sample_size
from causal_eval.estimators import (
    caliper_matching_att,
    cross_fitted_dr_ate,
    inverse_probability_weights,
    naive_difference,
    overlap_weighted_ate,
    propensity_scores,
    stabilized_ipw_ate,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

In [ ]:
data_path = project_root / "data" / "raw" / "lalonde.csv"
if not data_path.exists():
    raise FileNotFoundError("Run `make demo` or `scripts/download_data.py` first.")

df = pd.read_csv(data_path).rename(columns={"treat": "treatment", "re78": "outcome"})
df["black"] = (df["race"] == "black").astype(int)
df["hispan"] = (df["race"] == "hispan").astype(int)

feature_cols = ["age", "educ", "black", "hispan", "married", "nodegree", "re74", "re75"]
features = df[feature_cols].to_numpy().astype(float)
treatment = df["treatment"].to_numpy().astype(int)
outcome = df["outcome"].to_numpy().astype(float)

df.head()

## Propensity specification comparison

The repo chooses the propensity specification based on **post-weight balance**, not just fit metrics.

In [ ]:
spec_rows = []
for nonlinear in [False, True]:
    scores = propensity_scores(features, treatment, nonlinear=nonlinear)
    weights = inverse_probability_weights(treatment, scores, stabilized=True, trim=0.01)
    balance = covariate_balance_table(df, feature_cols, "treatment", weights=weights)
    spec_rows.append(
        {
            "specification": "nonlinear logistic" if nonlinear else "linear logistic",
            "max_abs_smd": balance["standardized_mean_diff"].abs().max(),
            "mean_abs_smd": balance["standardized_mean_diff"].abs().mean(),
            "ess": effective_sample_size(weights),
        }
    )

spec_table = pd.DataFrame(spec_rows).sort_values(["max_abs_smd", "mean_abs_smd"])
spec_table

In [ ]:
use_nonlinear_propensity = spec_table.iloc[0]["specification"] == "nonlinear logistic"
scores = propensity_scores(features, treatment, nonlinear=use_nonlinear_propensity)
stabilized_weights = inverse_probability_weights(treatment, scores, stabilized=True, trim=0.01)
overlap_weights = np.where(treatment == 1, 1.0 - scores, scores)

balance_before = covariate_balance_table(df, feature_cols, "treatment")
balance_after_sipw = covariate_balance_table(df, feature_cols, "treatment", weights=stabilized_weights)
balance_after_overlap = covariate_balance_table(df, feature_cols, "treatment", weights=overlap_weights)

balance_before.merge(balance_after_sipw, on="covariate", suffixes=("_before", "_after_sipw")).merge(
    balance_after_overlap.rename(columns={"standardized_mean_diff": "standardized_mean_diff_overlap"}),
    on="covariate",
)

## Estimator comparison

This is the same logic used in the generated report, but shown interactively.

In [ ]:
estimator_table = pd.DataFrame(
    {
        "estimator": [
            "naive difference",
            "caliper matching ATT",
            "stabilized IPW ATE",
            "overlap-weighted ATE",
            "cross-fitted DR ATE",
        ],
        "estimate": [
            naive_difference(outcome, treatment),
            caliper_matching_att(outcome, treatment, scores, caliper=0.2),
            stabilized_ipw_ate(outcome, treatment, scores),
            overlap_weighted_ate(outcome, treatment, scores),
            cross_fitted_dr_ate(outcome, treatment, features, nonlinear_propensity=use_nonlinear_propensity),
        ],
    }
)
estimator_table

## Placebo checks

A good causal workflow asks whether the method is inventing treatment effects on variables that happened **before** treatment.

In [ ]:
placebo_table = pd.DataFrame(
    {
        "pseudo_outcome": ["re74", "re75"],
        "cross_fitted_dr_estimate": [
            cross_fitted_dr_ate(df["re74"].to_numpy().astype(float), treatment, features, nonlinear_propensity=use_nonlinear_propensity),
            cross_fitted_dr_ate(df["re75"].to_numpy().astype(float), treatment, features, nonlinear_propensity=use_nonlinear_propensity),
        ],
    }
)
placebo_table

## Subgroup exploration

These are exploratory slices. They are useful for asking where support is weaker or where effect estimates move, but they are not a license to over-interpret noise.

In [ ]:
subgroup_masks = {
    "age >= median": df["age"] >= df["age"].median(),
    "no degree": df["nodegree"] == 1,
    "zero prior earnings": (df["re74"] == 0) & (df["re75"] == 0),
    "married": df["married"] == 1,
}

subgroup_rows = []
for label, mask in subgroup_masks.items():
    subset = df.loc[mask].copy()
    if subset["treatment"].nunique() < 2 or len(subset) < 80:
        continue
    subgroup_rows.append(
        {
            "subgroup": label,
            "n": len(subset),
            "cross_fitted_dr_estimate": cross_fitted_dr_ate(
                subset["outcome"].to_numpy().astype(float),
                subset["treatment"].to_numpy().astype(int),
                subset[feature_cols].to_numpy().astype(float),
                nonlinear_propensity=use_nonlinear_propensity,
            ),
        }
    )

pd.DataFrame(subgroup_rows)

## Generated artifacts

If you have already run `make demo`, the richer markdown reports and figures are available under `reports/`.

- `reports/causal_results.md`
- `reports/executive_memo.md`
- `reports/methods_appendix.md`
- `reports/figures/`

That split is intentional: the notebook is for exploration, while the report flow is for reproducible presentation.